## 1. Preliminaries 

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Loading of the Data 

In [ ]:
# Load the Excel file
file_path = "D:\\Downloads\\Healthcare Resource Utilization\\Data\\healthcare_dataset.xlsx" 
xls = pd.ExcelFile(file_path)

# Load each sheet into a separate DataFrame
patients_details_df = pd.read_excel(xls, sheet_name='Patients_details')
hospital_details_df = pd.read_excel(xls, sheet_name='Hospital_Details')
doctor_details_df = pd.read_excel(xls, sheet_name='Doctor_Details')
patients_data_df = pd.read_excel(xls, sheet_name='Patients_Data')

# Preview the data
print("✅ Patients Details:")
print(patients_details_df.head(), '\n')

print("✅ Hospital Details:")
print(hospital_details_df.head(), '\n')

print("✅ Doctor Details:")
print(doctor_details_df.head(), '\n')

print("✅ Patients Data:")
print(patients_data_df.head())


In [ ]:
# Preview the data
print("✅ Patients Details:")
print(patients_details_df.tail(), '\n')

print("✅ Hospital Details:")
print(hospital_details_df.tail(), '\n')

print("✅ Doctor Details:")
print(doctor_details_df.tail(), '\n')

print("✅ Patients Data:")
print(patients_data_df.tail())


## 3. Data Cleaning & Preparation 

In [ ]:
# Step 1: Merge patients_data with patients_details on P_ID
merged_df = pd.merge(patients_data_df, patients_details_df, on='P_ID', how='left')

# Step 2: Merge with doctor_details on D_ID
merged_df = pd.merge(merged_df, doctor_details_df, on='D_ID', how='left')

# Step 3: Merge with hospital_details on H_ID (note H_id in one table, H_ID in the other)
# If needed, rename for consistency
hospital_details_df.rename(columns={'H_id': 'H_ID'}, inplace=True)
merged_df = pd.merge(merged_df, hospital_details_df, on='H_ID', how='left')

In [ ]:
merged_df.head() #Top 5 Elements

In [ ]:
# Information of table
merged_df.info()

In [ ]:
# Describe the Table
merged_df.describe()

In [ ]:
# Get descriptive statistics for all columns, not just the numeric ones.
merged_df.describe(include="all")

In [ ]:
# Check for NaNs etc
merged_df.count()

In [ ]:
len(merged_df)

In [ ]:
# Datatypes
merged_df.dtypes

In [ ]:
# Length of dataframe
len(merged_df)

In [ ]:
# Columns
merged_df.columns

In [ ]:
# Standardize name columns in the merged DataFrame
merged_df['Name'] = merged_df['Name'].str.strip().str.title()
merged_df['Doctor'] = merged_df['Doctor'].str.strip().str.title()
merged_df['Hospital'] = merged_df['Hospital'].str.strip().str.title()

In [ ]:
merged_df.head()

###  c. Data Integrity Validation for Foreign Keys (P_ID, D_ID, H_ID) 
Identifying Mismatches and Foreign Key Issues Between P_ID, D_ID, and H_ID in Merged Data and Master Tables

In [ ]:
# Check for Unmatched Patient IDS (P_ID)
invalidid_pids = merged_df[merged_df['P_ID'].isin(~patients_details_df['P_ID'])]

# Check for Unmatched Patient IDS (D_ID)
invalidid_hids = merged_df[merged_df['H_ID'].isin(~hospital_details_df['H_ID'])]

# Check for Unmatched Patient IDS (D_ID)
invalidid_dids = merged_df[merged_df['D_ID'].isin(~doctor_details_df['D_ID'])]

print("invalidid_pids:", invalidid_pids)
print("invalidid_hids:", invalidid_hids)
print("invalidid_dids:", invalidid_dids)

### d. Handling Missing Values

Identifing and appropriately handling missing values in the dataset to prevent incomplete analysis or errors during visualization.

In [ ]:
# Summary of missing values
print("Missing Values Summary:")
print(merged_df.isnull().sum())

In [ ]:
# Drop rows where essntial patient info is missing (eg. Patient ID)
df = merged_df.dropna(subset=['P_ID'])

In [ ]:
df.count()

In [ ]:
# Fill Missing numerical values (eg., Age) with Median
df['Age'] = df['Age'].fillna(df['Age'].median())
df.count()

In [ ]:
df.isnull().sum()

In [ ]:
# Fill missing Categorical Values (e.g., Test Results) with mode
df['Test Results'] = df['Test Results'].fillna(df['Test Results'].mode()[0])
df.count()

In [ ]:
print("\nMissing Values After Cleaning:")
print(df.isnull().sum())

### e. Handling Duplicate Records 

Identifing and appropriately handling missing values in the dataset to prevent incomplete analysis or errors during visualization.

In [ ]:
# Check for duplicates (entire row match)
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows: {duplicate_count}")

In [ ]:
# Viewing duplicate rows 
duplicates = df[df.duplicated()]
print("\nDuplicate Rows Preview:")
print(duplicates.head())

In [ ]:
# Drop duplicate rows 
df = df.drop_duplicates()

In [ ]:
len(df)

In [ ]:
# drop based on a specific subset (e.g., Patient_ID + Date)
df1 = df.drop_duplicates(subset=['P_ID', 'Date of Admission'])
len(df1)

In [ ]:
print(f"\nData shape after removing duplicates: {df.shape}")

###  f. Converting Data Types 

Ensure all columns have correct data types for analysis.

In [ ]:
# Convert 'Date of Admission' & 'Date of Discharge' to datetime format
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'], errors='coerce', dayfirst=True) # we are telling Date start with Day  
df['Discharge Date'] = pd.to_datetime(df['Discharge Date'], errors='coerce', format='%d-%m-%y') #Customize the date format

df.dtypes

In [ ]:
print(df)

In [ ]:
#Converting format of date
custom_date = df['Date of Admission'].dt.strftime('%d-%m-%Y')

In [ ]:
custom_date

In [ ]:
custom_date.dtypes

In [ ]:
# Convert Billing Amount to numric(float)
df['Billing Amount'] = pd.to_numeric(df['Billing Amount'], errors = 'coerce')
df.dtypes

In [ ]:
# Convert P_ID to Str
df['P_ID'] = df['P_ID'].astype(str)
df.dtypes

In [ ]:
# Again Coverting P_ID to numeric
df['P_ID'] = pd.to_numeric(df['P_ID'])
df.dtypes

In [ ]:
# Convert 'Age' to integer
df['Age'] = pd.to_numeric(df['Age'], errors='coerce').astype('Int64')

In [ ]:
print(df.dtypes)

### g.  Creating New Derived Columns 

Creating useful new columns like Length of Stay or Billing Category.

In [ ]:
# Calculate length of stay
df['Length of Stay'] = (df['Discharge Date'] - df['Date of Admission']).dt.days

# Flag high billing amounts
df['High Bill Flag'] = df['Billing Amount'].apply(lambda x: 'High' if x > 30000 else 'Normal')

df.head()

### h. Mapping Categorical Values 

Mapping or encode categorical values for better readability or later modeling.

In [ ]:
# Distinct Values
df['Admission Type'].unique() 

In [ ]:
# Mapping Admission Type codes with given data
df['Admission Type Code'] = df['Admission Type'].map({'Elective' : 1,'Emergency' : 2, 'Urgent' :3})

In [ ]:
df.head()

### i. Final Abnormality Checks 

Verifing data consistency one last time — e.g., negative Length of Stay, future admission dates.

In [ ]:
#Checkig for negative Length of Stay
df[df['Length of Stay'] < 0]

In [ ]:
# Check if Admission Dates are in the future
df[df['Date of Admission'] > pd.Timestamp.today()]

## 4. Visulization Exploratory Data Analysis (EDA) 


### a. Univariate Analysis 

Univariate Analysis is the simplest form of data analysis where **only one variable** is analyzed at a time to understand its distribution, central tendency, spread, and underlying patterns.

### i. Histograms: Show frequency distribution of numeric variables.

In [ ]:
# Histogram for Age 
sns.histplot(df['Age'], bins = 20, kde = True) #Kernal Density Function
plt.title('Age Distribution')
plt.show()

### ii. Boxplots: Identify spread, central tendency, and outliers.

In [ ]:
# Boxplot for Billing Amount
sns.boxplot(x=df['Billing Amount'])
plt.title('Billing Amount Spread')
plt.show()

### iii. Count plots: Visualize counts (number of occurrences) for categorical variables.

In [ ]:
# Count Plot for Admission Type
sns.countplot(x='Admission Type', data=df)
plt.title('Admission Types Count')
plt.show()

### b. Bivariate Analysis

Bivariate Analysis is the **analysis of two variables simultaneously** to explore the **relationship, association, or correlation** between them and understand how one variable affects or relates to the other.

### i. Scatter plots: 

Show how **two continuous variables** relate.

In [ ]:
# Scatter Plot : Billing Amount vs. Length of Stay
sns.scatterplot(x='Length of Stay', y= 'Billing Amount', data = df.head(100))
plt.title('Billing Amount vs. Length of Stay')
plt.show()

### ii. Heatmaps: 

A heatmap is a data visualization technique that uses color gradients to represent the **magnitude or intensity of values in a two-dimensional matrix**.
It helps quickly identify **patterns, correlations, outliers, and areas of high or low concentration** within large datasets.

In [ ]:
# Step 1: Create Pivot Table
pivot_table = df.pivot_table(
    index='Admission Type',      # rows
    columns='Medical Condition', # columns
    values='Billing Amount',      # values
    aggfunc='mean'                # or 'sum', depending on what you want
)

# Step 2: Plot Heatmap
plt.figure(figsize=(8,5))
sns.heatmap(pivot_table, annot=True, fmt='.0f')
plt.title('Heatmap of Billing Amount by Admission Type and Medical Condition')
plt.xlabel('Medical Condition')
plt.ylabel('Admission Type')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.show()

### iii. Box Plot

Compares the distribution of a continuous variable across different categories (**categorical vs continuous**).

In [ ]:
# Boxplot: Billing by Admission Type
sns.boxplot(x='Admission Type', y='Billing Amount', data=df)
plt.title('Billing by Admission Type')
plt.show()

### iv. Bar plots: 

Visualize values (like **average, sum**) for categorical variables.

In [ ]:
plt.figure(figsize=(8,5))
ax = sns.barplot(x='Admission Type', y='Billing Amount', data=df, estimator=np.mean, palette='viridis')

# Data labels
for bar in ax.containers:
    ax.bar_label(bar, fmt='%.0f', label_type='edge', fontsize=10)

plt.title('Average Billing Amount by Admission Type')
plt.xlabel('Admission Type')
plt.ylabel('Average Billing Amount')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### v. Line Plot:

Shows the **trend or relationship between two continuous variables**, often across time.

In [ ]:
# Lineplot : Length of Stay vs. Billing Amount
sns.lineplot(x='Length of Stay', y ='Billing Amount', estimator = np.mean, data = df)
plt.title('Trend of Billing Amount over Length of Stay')
plt.show()

### c. Multivariate Analysis

Multivariate Analysis is the analysis of **more than two variables** simultaneously to understand complex **relationships, interactions, and combined effects among multiple variables** within a dataset.

### i. Grouped Boxplot :

 Compare distributions within categories

In [ ]:
# Grouped Boxplot: Billing Amount across Gender & Admission Type
sns.boxplot(x='Admission Type', y= 'Billing Amount', hue = 'Gender', data = df)
plt.title('Billing Amount by Admission Type & Gender')
plt.show()

### ii. Paired Plots:

Multiple scatter plots combined for selected variables.

In [ ]:
#Pariplot for Age, Billing Amount, Length of Stay

sns.pairplot(df.head(100), vars=['Age', 'Billing Amount', 'Length of Stay'], hue = 'Gender')
plt.suptitle('Multivariate Analysis with Pairplot', y=1.02) # y=1.02 → Adjusts the vertical position of the title.
plt.show()

### iii. Facet Grids: 

Create subplots based on categorical variables.

In [ ]:
# Create a FacetGrid: Billing Amount vs. Length of Stay, split by Gender
g = sns.FacetGrid(df.head(100), col='Admission Type', height=5, aspect=1)
g.map_dataframe(sns.scatterplot, x='Length of Stay', y='Billing Amount')

g.set_axis_labels('Length of Stay', 'Billing Amount')
g.set_titles('Gender: {col_name}')
g.fig.suptitle('Billing Amount vs. Length of Stay by Gender', y=1.05)
plt.show()

### d. Distribution Plots

Understand data distribution **patterns and proportions**

### i. KDE (Kernel Density Estimation): 

Smooth curve showing variable distribution.

In [ ]:
# KDE Plot for Billing Amount
sns.kdeplot(df['Billing Amount'], shade=True)
plt.title('Billing Amount Distribution - KDE')
plt.show()

### ii. Pie charts: 

Show percentage share of categories.

In [ ]:
# Pie Chart: Gender Distribution
df['Test Results'].value_counts().plot.pie(autopct='%1.1f%%', startangle=90)
plt.title('Test Results Distribution')
plt.ylabel('')
plt.show()

### e. Correlation Analysis

Measure the linear **relationship between continuous numerical variables** and identify highly correlated variables for insights or modeling.

### i. Correlation matrix & Heatmap

(values between -1 to 1)

In [ ]:
# Correlation Heatmap

corr = df[['Age', 'Billing Amount', 'Length of Stay']].corr() 
sns.heatmap(corr, annot = True, cmap = 'coolwarm')
plt.title('Correlation Heatmap')
plt.show()